# LlamaIndex와 AgentCore Memory - 투자 포트폴리오 Advisor(단기 메모리)

## 소개

이 Notebook에서는 Amazon Bedrock AgentCore Memory 기능을 LlamaIndex와 통합하여 투자 포트폴리오 Advisor를 만드는 방법을 살펴봅니다. 하나의 고객 상담 세션 안에서 **단기 메모리**를 유지하여 금융 자문 세션 전반에 걸쳐 고객 프로필, 포트폴리오 보유 자산, 시장 분석, 투자 추천을 기억하도록 하는 데 중점을 둡니다.

## 아키텍처 개요

![LlamaIndex AgentCore Short-Term Memory Architecture](LlamaIndex-AgentCore-STM-Arch.png)

## 튜토리얼 세부 정보

| 정보         | 세부 정보                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화 메모리                                                |
| Agent 사용 사례       | 투자 포트폴리오 Advisor                                                     |
| Agentic Framework   | LlamaIndex                                                                       |
| LLM 모델           | Anthropic Claude 3.7 Sonnet                                                       |
| 튜토리얼 구성 요소 | AgentCore Short-term Memory, LlamaIndex Agent, 금융 분석 도구         |
| 예제 난이도  | 중급                                                                     |

다음 내용을 학습합니다.
- 금융 자문 데이터를 위한 AgentCore Memory 생성
- 투자 workflow에 LlamaIndex 기본 메모리 통합 사용
- 포트폴리오 분석을 위한 금융 전용 도구 구축
- 단일 자문 세션 내에서 금융 컨텍스트 유지
- 메모리 경계 및 세션 격리 테스트

## 시나리오 배경

이 예제에서는 금융 Advisor가 단일 자문 세션 안에서 고객 포트폴리오를 분석하고 위험 지표를 평가하며 투자 추천을 제공하도록 돕는 "투자 포트폴리오 Advisor"를 만듭니다. Advisor는 AgentCore Memory를 사용하여 상담 전반에 걸쳐 고객 프로필, 포트폴리오 보유 자산, 시장 조사, 투자 분석에 관한 컨텍스트를 유지합니다.

## 사전 요구 사항

- Python 3.10 이상
- 적절한 권한이 있는 AWS 계정
- AgentCore Memory 권한이 있는 AWS IAM 역할:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock 모델에 대한 액세스

## 1단계: 종속성 설치 및 설정

In [ ]:
# 필요한 라이브러리 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3

In [ ]:
# 필요한 구성 요소 가져오기
from bedrock_agentcore.memory import MemoryClient
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

## 2단계: AgentCore Memory 구성

투자 Advisor에서 사용할 AgentCore Memory 리소스를 생성하거나 가져옵니다.

In [ ]:
# AgentCore Memory 리소스 생성
region = os.getenv("AWS_REGION", "us-east-1")
client = MemoryClient(region_name=region)

try:
    response = client.create_memory_and_wait(
        name=f"InvestmentAdvisorShortTerm_{int(datetime.now().timestamp())}",
        description="Investment portfolio advisor short-term memory for single session context",
        strategies=[],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10,
    )
    memory_id = response["id"]
    print(f"✅ Created AgentCore Memory: {memory_id}")
except Exception as e:
    print(f"❌ Error creating memory: {e}")
    memory_id = "your-memory-id-here"  # 기존 Memory ID로 교체

## 3단계: 금융 분석 도구 구현

투자 자문 작업을 위한 전문 도구를 정의합니다.

In [ ]:
def profile_client_risk(client_name: str, risk_tolerance: str, time_horizon: str, investment_goals: str) -> str:
    """Profile client risk tolerance and investment objectives"""
    print(f"👤 Client profile: {client_name} ({risk_tolerance} risk, {time_horizon} horizon)")
    return f"Profiled client: {client_name}"


def analyze_portfolio_holdings(portfolio_value: str, asset_allocation: str, top_holdings: str) -> str:
    """Analyze current portfolio holdings and allocation"""
    print(f"📊 Portfolio analysis: ${portfolio_value} total value, allocation: {asset_allocation}")
    return f"Analyzed portfolio worth ${portfolio_value}"


def calculate_risk_metrics(var_95: str, sharpe_ratio: str, beta: str, volatility: str) -> str:
    """Calculate portfolio risk metrics and performance indicators"""
    print(f"📈 Risk metrics: VaR 95% {var_95}, Sharpe {sharpe_ratio}, Beta {beta}, Vol {volatility}")
    return "Calculated risk metrics for portfolio"


def research_market_sector(sector: str, outlook: str, key_drivers: str, recommendation: str) -> str:
    """Research market sector with outlook and investment recommendation"""
    print(f"🔍 Sector research: {sector} - {outlook} outlook ({recommendation})")
    return f"Researched {sector} sector"


def generate_investment_recommendation(security: str, action: str, rationale: str, target_allocation: str) -> str:
    """Generate investment recommendation with rationale"""
    print(f"💡 Investment rec: {action} {security} (target: {target_allocation})")
    return f"Generated recommendation: {action} {security}"


def check_regulatory_compliance(rule_type: str, compliance_status: str, notes: str) -> str:
    """Check regulatory compliance for investment recommendations"""
    print(f"⚖️ Compliance check: {rule_type} - {compliance_status}")
    return f"Checked compliance: {rule_type}"


# Agent용 도구 객체 생성
financial_tools = [
    FunctionTool.from_defaults(fn=profile_client_risk),
    FunctionTool.from_defaults(fn=analyze_portfolio_holdings),
    FunctionTool.from_defaults(fn=calculate_risk_metrics),
    FunctionTool.from_defaults(fn=research_market_sector),
    FunctionTool.from_defaults(fn=generate_investment_recommendation),
    FunctionTool.from_defaults(fn=check_regulatory_compliance),
]

## 4단계: LlamaIndex Agent 구현

단기 메모리 컨텍스트를 사용하는 투자 Advisor Agent를 생성합니다.

In [ ]:
# 단기 메모리 구성(단일 세션)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

# 단일 세션용 메모리 컨텍스트 생성
context = AgentCoreMemoryContext(
    actor_id="financial-advisor",
    memory_id=memory_id,
    session_id="advisory-session-today",  # 전체 과정에서 동일한 세션 사용
    namespace="/investment-advisory/",
)

# AgentCore Memory 및 LLM 초기화
agentcore_memory = AgentCoreMemory(context=context)
llm = BedrockConverse(model=MODEL_ID)

# 투자 Advisor Agent 생성
investment_agent = FunctionAgent(tools=financial_tools, llm=llm, verbose=True)

print("✅ Investment Portfolio Advisor with short-term memory is ready!")

## 5단계: 단기 메모리 기능 테스트

종합 고객 자문 세션을 통해 투자 Advisor의 단기 메모리를 테스트해 보겠습니다.

### 테스트 1: 고객 등록 및 위험 성향 분석

In [ ]:
# 고객 세부 정보로 자문 세션 초기화
response = await investment_agent.run(
    "I'm Financial Advisor Michael Chen meeting with client 'Robert Johnson'. "
    "Profile client risk: 'Robert Johnson' with 'moderate' risk tolerance, '15 years' time horizon, "
    "and investment goals 'retirement planning, wealth preservation, moderate growth'.",
    memory=agentcore_memory,
)

print("🎯 Client Onboarding:")
print(response)

### 테스트 2: 포트폴리오 보유 자산 분석

In [ ]:
# 현재 포트폴리오 구성 분석
response = await investment_agent.run(
    "Analyze portfolio holdings: portfolio value '$2,500,000', asset allocation '60% stocks, 35% bonds, 5% cash', "
    "top holdings 'AAPL 8%, MSFT 7%, SPY 12%, BND 15%, VTIAX 10%'.",
    memory=agentcore_memory,
)

print("📊 Portfolio Analysis:")
print(response)

### 테스트 3: 위험 지표 계산

In [ ]:
# 종합 위험 지표 계산
response = await investment_agent.run(
    "Calculate risk metrics: VaR 95% '-$125,000', Sharpe ratio '1.15', Beta '0.85', volatility '12.3%'. "
    "Portfolio shows moderate risk profile with good risk-adjusted returns.",
    memory=agentcore_memory,
)

print("📈 Risk Metrics:")
print(response)

### 테스트 4: 고객 프로필 회상

In [ ]:
# 고객 정보 및 포트폴리오 회상 테스트
response = await investment_agent.run(
    "What client am I advising? What are their risk tolerance, investment goals, and current portfolio value?",
    memory=agentcore_memory,
)

print("🧠 Client Profile Recall:")
print(response)
print("\n✅ Expected: Robert Johnson, moderate risk, 15yr horizon, $2.5M portfolio, retirement planning")

### 테스트 5: 시장 섹터 조사

In [ ]:
# 잠재적 기회를 찾기 위한 기술 섹터 조사
response = await investment_agent.run(
    "Research market sector: 'Technology' with 'positive' outlook, key drivers 'AI adoption, cloud growth, digital transformation', "
    "recommendation 'overweight - increase allocation by 5%'.",
    memory=agentcore_memory,
)

print("🔍 Technology Sector Research:")
print(response)

# 분산 투자를 위한 헬스케어 섹터 조사
response = await investment_agent.run(
    "Research market sector: 'Healthcare' with 'neutral' outlook, key drivers 'aging demographics, drug innovation, regulatory changes', "
    "recommendation 'maintain - current allocation appropriate'.",
    memory=agentcore_memory,
)

print("🔍 Healthcare Sector Research:")
print(response)

### 테스트 6: 투자 추천

In [ ]:
# 구체적인 투자 추천 생성
response = await investment_agent.run(
    "Generate investment recommendation: security 'QQQ (Nasdaq ETF)', action 'BUY', "
    "rationale 'increase tech exposure per sector research, aligns with moderate risk profile', target allocation '8%'.",
    memory=agentcore_memory,
)

print("💡 QQQ Investment Recommendation:")
print(response)

response = await investment_agent.run(
    "Generate investment recommendation: security 'VGIT (Intermediate Treasury ETF)', action 'REDUCE', "
    "rationale 'rebalance to fund tech allocation, maintain duration risk management', target allocation '12%'.",
    memory=agentcore_memory,
)

print("💡 VGIT Rebalancing Recommendation:")
print(response)

### 테스트 7: 위험 지표 회상 및 분석

In [ ]:
# 위험 지표 기억 및 해석 테스트
response = await investment_agent.run(
    "What were Robert's current risk metrics? How does the Sharpe ratio and Beta align with his moderate risk tolerance?",
    memory=agentcore_memory,
)

print("📊 Risk Metrics Analysis:")
print(response)
print("\n✅ Expected: VaR -$125K, Sharpe 1.15, Beta 0.85, Vol 12.3% - good for moderate risk")

### 테스트 8: 규제 준수 확인

In [ ]:
# 추천의 규제 준수 여부 확인
response = await investment_agent.run(
    "Check regulatory compliance: rule type 'Fiduciary Duty - Best Interest', compliance status 'COMPLIANT', "
    "notes 'recommendations align with client risk profile and investment objectives'.",
    memory=agentcore_memory,
)

print("⚖️ Fiduciary Compliance:")
print(response)

response = await investment_agent.run(
    "Check regulatory compliance: rule type 'Portfolio Concentration Limits', compliance status 'COMPLIANT', "
    "notes 'no single position exceeds 15%, sector allocation within guidelines'.",
    memory=agentcore_memory,
)

print("⚖️ Concentration Compliance:")
print(response)

### 테스트 9: 투자 근거 종합

In [ ]:
# 통합 투자 추론 테스트
response = await investment_agent.run(
    "Based on my sector research and Robert's profile, why did I recommend increasing QQQ allocation? "
    "How does this align with his risk tolerance and investment goals?",
    memory=agentcore_memory,
)

print("🤔 Investment Rationale:")
print(response)
print("\n✅ Expected: Tech sector positive outlook + moderate risk tolerance + 15yr horizon = QQQ increase")

In [ ]:
# 종합 자문 세션 요약
response = await investment_agent.run(
    "Provide a complete advisory summary: client profile, current portfolio metrics, sector research findings, "
    "investment recommendations, and compliance status. Include rationale for all recommendations.",
    memory=agentcore_memory,
)

print("📋 Complete Advisory Summary:")
print(response)
print(
    "\n✅ Expected: Full session details with Robert's profile, $2.5M portfolio, tech/healthcare research, QQQ/VGIT recs"
)

## 6단계: 세션 경계 테스트

별도의 세션을 생성하여 단기 메모리의 경계를 테스트해 보겠습니다.

In [ ]:
# 별도의 세션 컨텍스트 생성
new_session_context = AgentCoreMemoryContext(
    actor_id="financial-advisor",
    memory_id=memory_id,
    session_id="different-advisory-session",  # 서로 다른 세션 ID
    namespace="/investment-advisory/",
)

new_session_memory = AgentCoreMemory(context=new_session_context)

# 메모리 격리 테스트
response = await investment_agent.run(
    "What clients am I advising today? What portfolio values and investment recommendations have I made?",
    memory=new_session_memory,
)

print("🚧 Session Boundary Test (Different Session):")
print(response)
print("\n✅ Expected: Limited or no recall from previous session (short-term memory boundary)")

In [ ]:
# 지속성을 검증하기 위해 원래 세션으로 복귀
response = await investment_agent.run(
    "Back in my original session - what were Robert Johnson's exact risk metrics and my QQQ recommendation?",
    memory=agentcore_memory,  # 원래 세션 메모리
)

print("🔄 Original Session Return:")
print(response)
print("\n✅ Expected: Full recall of Sharpe 1.15, Beta 0.85, QQQ BUY 8% allocation")

## 🧪 자동 테스트 검증
다음 셀을 실행하여 메모리 통합이 올바르게 작동하는지 검증합니다.

In [ ]:
# 검증 함수를 인라인으로 정의
class TestValidator:
    def __init__(self):
        self.results = {}

    def validate_memory_recall(self, response):
        """에이전트가 세션 앞부분의 정보를 기억하는지 확인합니다."""
        # 실질적인 응답인지 확인("I don't know"만 있는 응답 제외)
        has_content = len(response) > 50
        # 메모리 관련 표현 확인
        has_memory_indicators = any(
            word in response.lower()
            for word in [
                "earlier",
                "mentioned",
                "discussed",
                "previously",
                "you",
                "we",
                "our",
            ]
        )
        return "✅ PASS" if (has_content and has_memory_indicators) else "❌ FAIL"

    def validate_session_memory(self, response):
        """에이전트가 세션 내 컨텍스트를 유지하는지 확인합니다."""
        has_memory_content = len(response) > 100 and any(
            word in response.lower()
            for word in [
                "previous",
                "earlier",
                "mentioned",
                "discussed",
                "before",
                "already",
            ]
        )
        return "✅ PASS" if has_memory_content else "❌ FAIL"

    def validate_cross_reference(self, response):
        """에이전트가 현재 질의를 이전 컨텍스트와 연결할 수 있는지 확인합니다."""
        # 연결 표현 확인
        connecting_words = [
            "relate",
            "connection",
            "previous",
            "earlier",
            "discussed",
            "mentioned",
            "context",
            "based on",
            "as we",
            "as i",
        ]
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "✅ PASS" if (has_connection and has_substance) else "❌ FAIL"

    def run_validation_summary(self, test_results):
        print("🧪 COMPREHENSIVE TEST VALIDATION SUMMARY")
        print("=" * 60)

        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "PASS" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0

        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")

        print("=" * 60)
        print(f"📊 Overall Pass Rate: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")

        if pass_rate >= 80:
            print("✅ EXCELLENT: Memory integration working correctly!")
        elif pass_rate >= 60:
            print("⚠️  GOOD: Most memory features working, some issues to investigate")
        else:
            print("❌ NEEDS ATTENTION: Memory integration has significant issues")

        return pass_rate


validator = TestValidator()
print("✅ Validation functions loaded!")

In [ ]:
# 모든 검증 테스트 실행
test_results = {}

# 테스트 1: 메모리 회상 - Agent가 논의한 내용을 기억하는가?
response1 = await investment_agent.run("What have we discussed so far in this session?", memory=agentcore_memory)
test_results["Memory Recall"] = validator.validate_memory_recall(str(response1))
print(f"Response 1 length: {len(str(response1))} chars")

# 테스트 2: 세션 메모리 - Agent가 컨텍스트를 유지하는가?
response2 = await investment_agent.run("What did we talk about earlier?", memory=agentcore_memory)
test_results["Session Memory"] = validator.validate_session_memory(str(response2))
print(f"Response 2 length: {len(str(response2))} chars")

# 테스트 3: 상호 참조 기능 - Agent가 이전 컨텍스트와 연결할 수 있는가?
response3 = await investment_agent.run("How does this relate to what we discussed before?", memory=agentcore_memory)
test_results["Cross Reference"] = validator.validate_cross_reference(str(response3))
print(f"Response 3 length: {len(str(response3))} chars")

# 결과 표시
validator.run_validation_summary(test_results)

### 테스트 10: 종합 자문 요약

## 요약

이 Notebook에서는 다음 내용을 구현했습니다.

✅ **단기 메모리 통합**: LlamaIndex에서 AgentCore Memory를 사용하여 세션 범위의 투자 자문 제공

✅ **금융 전용 도구**: 고객 프로파일링, 포트폴리오 분석, 위험 지표, 투자 추천

✅ **투자 추론**: Advisor가 고객 프로필, 시장 조사, 추천 근거를 기억

✅ **위험 관리**: 종합적인 위험 지표 추적 및 규제 준수 확인

✅ **세션 경계**: 서로 다른 고객 자문 세션 간의 메모리 격리

✅ **규제 준수**: 신탁 의무 및 투자 가이드라인 준수

투자 포트폴리오 Advisor는 단기 메모리를 통해 하나의 고객 세션 안에서 종합적인 금융 자문을 제공하는 동시에 서로 다른 고객 상담 사이의 경계를 명확히 유지하는 방법을 보여 줍니다.

## 리소스 정리

이 Notebook에서 사용한 리소스를 정리하기 위해 Memory를 삭제합니다.

In [ ]:
# AgentCore Memory 리소스 정리
try:
    client.delete_memory(memory_id)
    print(f"✅ Successfully deleted memory: {memory_id}")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")